# Figure 2 — picking a target

Specificity headroom per receptor, the extracellular cysteine architecture, and glycan masking potential. The first panel is the actionable one: receptors with low identity to their nearest neighbour are the clean targets.

## Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import style

style.apply()
if not style.available():
    print("Arial not found, matplotlib fell back to DejaVu Sans")

RESULTS, FIGS = ROOT / "results", ROOT / "figures"
FIGS.mkdir(exist_ok=True)
PAL = style.PALETTE
SEGMENTS = ["N-term", "ECL1", "ECL2", "ECL3"]

## Load

In [ ]:
nn = pd.read_csv(RESULTS / "nearest_neighbours.csv")
dis = pd.read_csv(RESULTS / "disulfide_audit.csv")
seg = pd.read_csv(RESULTS / "segments_long.csv")
N_LABEL = 8
nn.head()

## Panels

In [ ]:
fig, ax = plt.subplot_mosaic([["nn", "nn"], ["cys", "glyc"]],
                             figsize=(10.5, 7.8), constrained_layout=True)

a = ax["nn"]
for i, s in enumerate(SEGMENTS):
    d = nn[nn.segment == s].sort_values("nearest_pid")
    y = d["nearest_pid"].values
    a.scatter(np.random.normal(i, 0.09, len(y)), y, s=7, alpha=0.5,
              color=PAL[s], edgecolors="none")
    for _, r in d.head(N_LABEL).iterrows():
        a.annotate(r["entry_name"].split("_")[0], (i + 0.16, r["nearest_pid"]),
                   fontsize=6, color="0.25")
a.set_xticks(range(len(SEGMENTS)))
a.set_xticklabels(SEGMENTS)
a.set_ylabel("identity to closest other receptor")
a.set_title("Specificity headroom (low = isolated = clean target)")

a = ax["cys"]
bars = {
    "Cys3.25 present": dis["has_C3x25"].mean(),
    "Cys in ECL2": (dis["n_cys_ECL2"] >= 1).mean(),
    "Canonical pair": dis["canonical_disulfide"].mean(),
    "2+ Cys in ECL2": dis["extra_ECL2_cys"].mean(),
    "N-term + ECL3 Cys": dis["possible_Nterm_ECL3_bond"].mean(),
}
a.barh(list(bars)[::-1], [v * 100 for v in list(bars.values())[::-1]],
       color=PAL["ECL2"], alpha=0.85)
a.set_xlabel("% of receptors")
a.set_xlim(0, 100)
a.set_title("Extracellular cysteine architecture")

a = ax["glyc"]
for s in SEGMENTS:
    d = seg[seg.segment == s]
    a.scatter(d["length"], d["n_sequon"] + np.random.normal(0, 0.06, len(d)),
              s=7, alpha=0.45, color=PAL[s], label=s, edgecolors="none")
a.set_xscale("log")
a.set_xlabel("length (aa)")
a.set_ylabel("N-X-S/T sequons")
a.set_title("Glycan masking potential")
a.legend()

fig.savefig(FIGS / "02_targets.png")